# Le choix de la fenêtre d'observation

- **Objectif :** trouver l'intervalle optimal nécessaire pour prédire le décrochage scolaire.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from config import (
    CLEAN_ACTIVITES,
    CLEAN_EVALUATIONS,
    CLEAN_INSCRIPTIONS,
)

## 1. Préparation des données

### 1.1 Chargement des données

In [ ]:
activities_df = pd.read_parquet(CLEAN_ACTIVITES)
evaluations_df = pd.read_parquet(CLEAN_EVALUATIONS)
inscriptions_df = pd.read_parquet(CLEAN_INSCRIPTIONS)

### 1.2 Jointure des tables d'événements

Pour exprimer chaque événement en jours écoulés depuis le début de l'inscription plutôt qu'en date absolue.

In [ ]:
cols_ins = ['id_inscription', 'date_debut', 'date_debut_programme', 'date_annulation', 'semestre', 'niveau_scolaire']
activities_merged = activities_df.merge(inscriptions_df[cols_ins], on='id_inscription', how='left')
evaluations_merged = evaluations_df.merge(inscriptions_df[cols_ins], on='id_inscription', how='left')

In [ ]:
activities_merged['days_since_start'] = (activities_merged['horodatage'] - activities_merged['date_debut_programme']).dt.days
evaluations_merged['days_since_start'] = (evaluations_merged['date_soumission'] - evaluations_merged['date_debut_programme']).dt.days

print("Activities days_since_start description:")
print(activities_merged['days_since_start'].describe())
print("\nEvaluations days_since_start description:")
print(evaluations_merged['days_since_start'].describe())

## 2. Visualisation

In [ ]:
sns.histplot(data=activities_merged, x='days_since_start', hue='semestre', kde=True)
plt.xlabel('Days since programme start (T_0)')
plt.ylabel('Activity Frequency')
plt.show()

In [ ]:
sns.histplot(data=evaluations_merged, x='days_since_start', hue='semestre', kde=True)
plt.xlabel('Days since programme start (T_0)')
plt.ylabel('Evaluation Frequency')
plt.show()

In [ ]:
print("Activités pré-rentrée (days_since_start < 0) :", (activities_merged['days_since_start'] < 0).sum())
print("Évaluations pré-rentrée (days_since_start < 0) :", (evaluations_merged['days_since_start'] < 0).sum())

In [ ]:
print(activities_merged.groupby('semestre', observed=False)['days_since_start'].describe())
print(evaluations_merged.groupby('semestre', observed=False)['days_since_start'].describe())

## 3. Temps avant abandon

### 3.1 Distribution du temps de vie des inscriptions "Abandon"

In [ ]:
student_churn = inscriptions_df[inscriptions_df['resultat_final'] == 'Abandon'].copy()
student_churn['lifespan'] = (student_churn['date_annulation'] - student_churn['date_debut_programme']).dt.days
print(student_churn.groupby('semestre', observed=False)['lifespan'].describe())
print("\nQuantiles globaux de lifespan (jours depuis T_0) :")
print(student_churn['lifespan'].quantile([0.05, 0.10, 0.15, 0.20, 0.25, 0.50, 0.75]))

### 3.2 Visualisation

In [ ]:
sns.histplot(data=student_churn, x='lifespan', hue='semestre', kde=True)
plt.xlabel('Lifespan from T_0 (days)')
plt.ylabel('Frequency')
plt.title('Distribution of Lifespan for Churned Students by Semester')
plt.show()

## 4. Sélection des candidats T_obs

### 4.1 Vérification de la troncature liée à l'horizon des données

Un T_obs trop long pourrait dépasser la date limite d'extraction des données pour les inscriptions les plus tardives, ce qui fausserait les features de fenêtre pour ces cas sans lien avec leur comportement réel.

In [ ]:
data_cutoff = activities_df['horodatage'].max()
print("Date max d'activité :", data_cutoff)

In [ ]:
students_df = inscriptions_df.copy()
for T_obs in [15, 30, 45, 60, 90]:
    students_df['window_end'] = students_df['date_debut_programme'] + pd.Timedelta(days=T_obs)
    truncated = (students_df['window_end'] > data_cutoff).sum()
    print(f"T_obs: {T_obs} days, Truncated: {truncated} students")

### 4.2 Diagnostic de couverture et de risque de fuite par candidat

Pour chaque T_obs candidat (30, 60, 90 jours) : part des inscriptions sans activité/évaluation dans la fenêtre, et part des abandons déjà partis avant la fermeture de la fenêtre (indicateur direct de risque de fuite temporelle).

In [ ]:
total_inscriptions = inscriptions_df['id_inscription'].nunique()
results = []
for T_obs in [15, 30, 45, 60, 90]:
    # Inscriptions encore actives à T_obs (n'ayant pas déjà abandonné à <= T_obs)
    active_ins = inscriptions_df[
        inscriptions_df['date_annulation'].isna()
        | ((inscriptions_df['date_annulation'] - inscriptions_df['date_debut_programme']).dt.days > T_obs)
    ]
    n_active = len(active_ins)

    activities_window = activities_merged[
        (activities_merged['days_since_start'] <= T_obs)
        & (activities_merged['id_inscription'].isin(active_ins['id_inscription']))
    ]
    activity_summary = activities_window.groupby('id_inscription').agg(
        total_activities=('id_activite', 'count'),
        total_volume=('volume', 'sum'),
    )

    evaluations_window = evaluations_merged[
        (evaluations_merged['days_since_start'] <= T_obs)
        & (evaluations_merged['id_inscription'].isin(active_ins['id_inscription']))
    ]
    evaluation_summary = evaluations_window.groupby('id_inscription').agg(
        total_evaluations=('id_evaluation', 'count')
    )

    n_activity = activity_summary.shape[0]
    n_evaluation = evaluation_summary.shape[0]

    results.append({
        'T_obs': T_obs,
        'active_students': n_active,
        'no activity % (active)': round(100 * (1 - n_activity / n_active), 2),
        'no evaluation % (active)': round(100 * (1 - n_evaluation / n_active), 2),
        'median activity': activity_summary['total_activities'].median(),
        'median evaluation': evaluation_summary['total_evaluations'].median(),
        'already churned %': round(100 * (student_churn['lifespan'] <= T_obs).mean(), 2),
    })

diagnostics_df = pd.DataFrame(results)
display(diagnostics_df)

In [ ]:
# Cadence des évaluations
evaluations_merged['deadline_from_start'] = (
    evaluations_merged['date_echeance'] - evaluations_merged['date_debut_programme']
).dt.days
first_eval_deadline = evaluations_merged.groupby('id_inscription')['deadline_from_start'].min()
print("Première échéance d'évaluation (jours depuis T_0) :")
print(first_eval_deadline.describe())

## 5. Fiabilité des dates d'abandon et écart
 $\Delta_i = T_i^{\text{cancel}} - T_i^{\text{last\_interaction}}$

In [ ]:
from config import (
    RAW_INSCRIPTIONS,
    RAW_PROGRAMMES,
    RAW_ACTIVITES,
    RAW_EVALUATIONS,
)

# Chargement des tables brutes pour distinguer les dates d'annulation observées des dates manquantes
raw_ins = pd.read_csv(
    RAW_INSCRIPTIONS,
    parse_dates=['date_debut', 'date_annulation'],
    dtype={'resultat_final': 'category'},
)
raw_prog = pd.read_csv(RAW_PROGRAMMES, dtype={'filiere': 'string'})
raw_act = pd.read_csv(
    RAW_ACTIVITES,
    parse_dates=['horodatage'],
    dtype={'type_activite': 'category'},
)
raw_eval = pd.read_csv(
    RAW_EVALUATIONS,
    parse_dates=['date_echeance', 'date_soumission'],
    dtype={'type_evaluation': 'category'},
)

# Dérivation de T_0 (date_debut_programme)
annees = raw_prog['annee_scolaire'].str.split('/', expand=True)
raw_prog['date_debut_programme'] = pd.to_datetime(
    np.where(
        raw_prog['semestre'] == 'Semestre 1',
        annees[0] + '-09-01',
        annees[1] + '-01-01',
    )
)

audit_ins = raw_ins.merge(
    raw_prog[['id_programme', 'niveau_scolaire', 'annee_scolaire', 'semestre', 'date_debut_programme']],
    on='id_programme',
    how='left',
)

# Dernière interaction (activité ou soumission d'évaluation valide) par inscription
valid_eval = raw_eval.merge(raw_ins[['id_inscription', 'date_debut']], on='id_inscription', how='left')
valid_eval = valid_eval[
    ~(
        (valid_eval['date_soumission'] < valid_eval['date_debut'])
        | (valid_eval['date_echeance'] < valid_eval['date_debut'])
    )
]

last_act_date = raw_act.groupby('id_inscription')['horodatage'].max().rename('last_act_date')
last_eval_date = valid_eval.groupby('id_inscription')['date_soumission'].max().rename('last_eval_date')

audit_ins = audit_ins.merge(last_act_date, on='id_inscription', how='left')
audit_ins = audit_ins.merge(last_eval_date, on='id_inscription', how='left')
audit_ins['last_interaction_date'] = audit_ins[['last_act_date', 'last_eval_date']].max(axis=1)

# Exclusion des 127 abandons pré-rentrée (date_annulation observée < T_0) et des 4 Echec avec date_annulation
pre_t0_cancel = audit_ins['date_annulation'].notna() & (
    audit_ins['date_annulation'] < audit_ins['date_debut_programme']
)
label_mismatch = audit_ins['date_annulation'].notna() & (audit_ins['resultat_final'] != 'Abandon')
audit_post_t0 = audit_ins[~(pre_t0_cancel | label_mismatch)].copy()

# Jours relatifs par rapport à T_0
audit_post_t0['cancel_day'] = (
    audit_post_t0['date_annulation'] - audit_post_t0['date_debut_programme']
).dt.days
audit_post_t0['last_interaction_day'] = (
    audit_post_t0['last_interaction_date'] - audit_post_t0['date_debut_programme']
).dt.days

# 1. Comptage des abandons confirmés vs sans date_annulation
abandons = audit_post_t0[audit_post_t0['resultat_final'] == 'Abandon'].copy()
abandons['has_cancel_date'] = abandons['date_annulation'].notna()

print("=== 1. Abandons post-T_0 : Date d'annulation observée vs manquante ===")
print(abandons['has_cancel_date'].value_counts())
print("\nPar semestre :")
print(pd.crosstab(abandons['semestre'], abandons['has_cancel_date'], margins=True))
print("\nInspecter les 14 abandons sans date_annulation :")
display(
    abandons.loc[
        ~abandons['has_cancel_date'],
        ['id_inscription', 'id_programme', 'semestre', 'niveau_scolaire', 'last_interaction_day'],
    ]
)

In [ ]:
# 2. Distribution de Delta_i = T_cancel - T_last_interaction sur les abandons confirmés (N = 1600)
confirmed_abandons = abandons[abandons['has_cancel_date']].copy()
confirmed_abandons['delta_days'] = (
    confirmed_abandons['cancel_day'] - confirmed_abandons['last_interaction_day']
)

print("=== 2. Distribution de Delta_i = T_cancel - T_last_interaction (en jours) ===")
print(confirmed_abandons['delta_days'].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))
print(f"\nCas où T_last_interaction > T_cancel (delta_days < 0) : {(confirmed_abandons['delta_days'] < 0).sum()} ({(confirmed_abandons['delta_days'] < 0).mean():.2%})")
print(f"Cas où |delta_days| <= 7 jours : {(confirmed_abandons['delta_days'].abs() <= 7).mean():.2%}")
print(f"Cas où delta_days > 30 jours : {(confirmed_abandons['delta_days'] > 30).mean():.2%}")

# 3. Comparaison du dernier jour d'interaction selon resultat_final
print("\n=== 3. Dernier jour d'interaction (depuis T_0) par resultat_final ===")
print(
    audit_post_t0.groupby('resultat_final', observed=False)['last_interaction_day'].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

## 6. Durée des programmes, fin de période à risque et table de survie par palier
 $t_k = 30k$

In [ ]:
# 1. Horizon temporel par semestre et niveau_scolaire (sur les non-abandons : Reussite + Echec)
persisted = audit_post_t0[audit_post_t0['resultat_final'] != 'Abandon'].copy()

prog_horizon = (
    persisted.groupby(['semestre', 'niveau_scolaire'], observed=True)['last_interaction_day']
    .agg(
        n_persisted='count',
        q10=lambda s: np.percentile(s, 10),
        q25=lambda s: np.percentile(s, 25),
        median='median',
        q75=lambda s: np.percentile(s, 75),
        q90=lambda s: np.percentile(s, 90),
        q95=lambda s: np.percentile(s, 95),
        max_day='max',
    )
    .round(1)
)
print("=== 1. Horizon du dernier jour d'interaction des étudiants persistants (Reussite/Echec) ===")
display(prog_horizon)

# Horizon des échéances d'évaluation (date_echeance - T_0) par semestre et niveau_scolaire
eval_prog = valid_eval.merge(
    audit_post_t0[['id_inscription', 'semestre', 'niveau_scolaire', 'date_debut_programme']],
    on='id_inscription',
    how='inner',
)
eval_prog['due_day'] = (eval_prog['date_echeance'] - eval_prog['date_debut_programme']).dt.days
eval_prog['sub_day'] = (eval_prog['date_soumission'] - eval_prog['date_debut_programme']).dt.days

deadline_horizon = (
    eval_prog.groupby(['semestre', 'niveau_scolaire'], observed=True)
    .agg(
        n_evals=('id_evaluation', 'count'),
        max_due_day=('due_day', 'max'),
        q95_sub_day=('sub_day', lambda s: np.percentile(s.dropna(), 95)),
        max_sub_day=('sub_day', 'max'),
    )
    .round(1)
)
print("\n=== 2. Horizon des évaluations (échéances et soumissions en jours depuis T_0) ===")
display(deadline_horizon)

In [ ]:
# 3. Table de risque longitudinale aux paliers t_k = 30k (sur les inscriptions à date connue : N = 7024)
known_cohort = audit_post_t0[
    (audit_post_t0['resultat_final'] != 'Abandon') | (audit_post_t0['date_annulation'].notna())
].copy()

checkpoints = [30, 60, 90, 120, 150, 180, 210, 240]
hazard_rows = []

for t_k in checkpoints:
    for sem in ['Semestre 1', 'Semestre 2']:
        sub = known_cohort[known_cohort['semestre'] == sem]
        # Encore non-annulé à t_k
        at_risk = sub[sub['cancel_day'].isna() | (sub['cancel_day'] > t_k)]
        # Abandon dans (t_k, t_k + 30]
        events_next_30d = at_risk[
            (at_risk['resultat_final'] == 'Abandon') & (at_risk['cancel_day'] <= t_k + 30)
        ]
        # Parmi les non-abandons à risque, combien ont encore une interaction après t_k ?
        persisted_at_risk = at_risk[at_risk['resultat_final'] != 'Abandon']
        active_after_tk = (persisted_at_risk['last_interaction_day'] > t_k).mean()

        hazard_rows.append({
            'semestre': sem,
            'checkpoint_tk': f"J+{t_k}",
            'window_predicted': f"({t_k}, {t_k+30}]",
            'n_at_risk': len(at_risk),
            'n_churn_next_30d': len(events_next_30d),
            'hazard_rate_%': round(100 * len(events_next_30d) / max(len(at_risk), 1), 2),
            'persisted_active_after_tk_%': round(100 * active_after_tk, 2),
        })

hazard_df = pd.DataFrame(hazard_rows)
print("=== 3. Table de survie et taux de risque (Hazard Rate) par palier de 30 jours ===")
display(hazard_df)

## Conclusion

- **Homogénéité inter-semestres :** Les distributions d'activité (`days_since_start` médian de 94 jours au Semestre 1 et 76 jours au Semestre 2) et de durée de vie avant abandon (`lifespan` médian de 80 jours au Semestre 1 et 83 jours au Semestre 2, soit 81 jours globalement) présentent une dynamique similaire entre les deux semestres[cite: 10, 11, 12].
- **Activité pré-rentrée ($t < T_0$) :** On observe 167 770 enregistrements d'activités et 487 soumissions d'évaluations réalisés entre la date d'inscription (`date_debut`) et le début officiel du programme (`date_debut_programme`, $T_0$). Ces interactions reflètent l'engagement initial pré-rentrée et sont conservées dans le calcul des indicateurs cumulés ($\le T_{\text{obs}}$).
- **Absence de troncature :** La date maximale d'extraction étant le `2025-08-30`, aucune inscription n'est tronquée par l'horizon des données pour $T_{\text{obs}} \in \{15, 30, 45, 60, 90\}$ jours (0 étudiant tronqué).
- **Cadence des évaluations :** Rapportée à la date officielle de début du programme ($T_0$), la première échéance d'évaluation (`first_eval_deadline`) a une médiane de **19 jours** (Q1 = 19 jours, Q3 = 25 jours). Ainsi, dès $T_{\text{obs}} = 30$ jours, plus de 75 % des programmes ont dépassé leur première échéance d'évaluation.
- **Arbitrage couverture vs départ précoce (choix de $T_{\text{obs}}$) :**
  - À **$T_{\text{obs}} = 15$ jours**, 83,98 % des étudiants actifs n'ont encore aucune évaluation (le premier jalon académique de 19 jours n'étant pas atteint).
  - Entre **15 et 30 jours**, la part d'étudiants actifs sans évaluation chute de **83,98 % à 23,24 %**, et seuls **1,28 %** n'ont pas d'activité enregistrée (médiane de 80 activités).
  - À **$T_{\text{obs}} = 60$ jours**, **42,11 %** des abandons ont déjà quitté la plateforme avant la fin de la fenêtre (et **53,17 %** à 90 jours), ce qui réduit fortement l'utilité opérationnelle d'un système d'alerte précoce.
- **Décision :** **$T_{\text{obs}} = 30$ jours** est retenu comme fenêtre d'observation principale. Ce seuil capture le premier cycle d'évaluations (76,76 % d'étudiants évalués) et l'engagement du premier mois tout en permettant d'intervenir sur **74,91 %** des futurs décrocheurs (avec $T_{\text{obs}} = 45$ jours comme variante de sensibilité).